In [13]:
import threading
import time
from pathlib import Path

import torch
import numpy as np
import tqdm

from diff_mfld.geodesic.geodesic_funcs import ExpMethod, LogMethod
from diff_mfld.geometry.metric import MetricField
from diff_mfld.mfld import ComputeMfld
from optim.subsolver_methods import SubsolverMethod
from optim.subsolvers.rgd import RiemGradDescentCfg
from optim.subsolvers.rtr import RiemTrustRegionCfg
from problem import create_problem

In [19]:
methods = [
    ((ExpMethod.IVP, LogMethod.BVP), "ivpbvp"),
    # ((ExpMethod.APPROX_O1, LogMethod.APPROX_O1), "o1"),
    # ((ExpMethod.APPROX_O2, LogMethod.APPROX_O2), "o2"),
    # ((ExpMethod.APPROX_O3, LogMethod.APPROX_O3), "o3"),
    # ((ExpMethod.APPROX_O2, LogMethod.APPROX_O1), "o2_o1"),
    # ((ExpMethod.APPROX_O3, LogMethod.APPROX_O1), "o3_o1"),
    # ((ExpMethod.APPROX_O3, LogMethod.APPROX_O2), "o3_o2"),
]

# linear scaling of the domain
max_domain_scaling = [
    1., 2., 3., # at 5 there are issues with solving logarithmic map at o3
]

# parameters for random starting positions
p0_mean = np.array([2., -5., -8.])
p0_covar = 2. ** 2 * np.eye(3)
num_p0 = 20

# cost minimized at this point
cost_center = np.array([-5., -1., 3.])


# the various metrics (describing the given curvature of the surface of optimization)

def euclid_metric(x1, x2, x3, scaling: float):
    metric = torch.eye(3)

    # pulls the metric "back" onto the scaled coordinate system
    factor = 1. / scaling ** 2
    metric = factor * metric # avoids in-place operation to keep pytorch gradients

    return metric


def scaled_metric(x1, x2, x3, scaling: float):
    # elements are assigned to this metric directly to preserve gradient history
    metric = torch.zeros((3, 3))
    metric[0, 0] = x1 ** 2 + 1.  # constant prevents degeneracy at origin
    metric[1, 1] = x2 ** 2 + 1.
    metric[2, 2] = x3 ** 2 + 1.

    # pulls the metric "back" onto the scaled coordinate system
    factor = 1. / scaling ** 2
    metric = factor * metric # avoids in-place operation to keep pytorch gradients

    return metric


def coupled_metric(x1, x2, x3, scaling: float):
    factor = 1. / scaling ** 2

    metric = torch.zeros((3, 3))
    metric[0, 0] = x1 ** 2 + 1.
    metric[0, 1] = 0.5 * x1 * x2
    metric[0, 2] = 0.5 * x1 * x3

    metric[1, 0] = 0.5 * x2 * x1
    metric[1, 1] = x2 ** 2 + 1.
    metric[1, 2] = 0.5 * x2 * x3

    metric[2, 0] = 0.5 * x3 * x1
    metric[2, 1] = 0.5 * x3 * x2
    metric[2, 2] = x3 ** 2 + 1.

    # pulls the metric "back" onto the scaled coordinate system
    factor = 1. / scaling ** 2
    metric = factor * metric # avoids in-place operation to keep pytorch gradients

    return metric


def asymmetric_metric(x1, x2, x3, scaling: float):
    factor = 1. / scaling ** 2

    metric = torch.zeros((3, 3))
    metric[0, 0] = x1 ** 2 + 1.
    metric[1, 1] = 0.5 * x1 ** 2 * x2 ** 2 + 1.
    metric[2, 2] = 0.25 * x1 ** 2 * x2 ** 2 * x3 ** 2 + 1.

    # pulls the metric "back" onto the scaled coordinate system
    factor = 1. / scaling ** 2
    metric = factor * metric # avoids in-place operation to keep pytorch gradients

    return metric


metric_funcs = [
    (euclid_metric, "euclid"),
    (scaled_metric, "scaled"),
    (coupled_metric, "coupled"),
    (asymmetric_metric, "asymmetric"),
]


In [15]:
# generates the random points and scales them

# "To Everything? To the great Question of Life, the Universe and everything?"
r = np.random.default_rng(42)
p_rand_starting = r.multivariate_normal(p0_mean, p0_covar, num_p0)
max_starting_norm = np.max(np.linalg.vector_norm(p_rand_starting, axis=1))

# scales the optimization functions and starting points
p_rand_starting /= max_starting_norm
cost_center /= max_starting_norm

In [16]:
# configure the (unconstrained) configurations
optimizers = [
    ((SubsolverMethod.RIEM_GRAD_DESCENT, RiemGradDescentCfg()), "rgd"),
    ((SubsolverMethod.RIEM_TRUST_REGION, RiemTrustRegionCfg()), "rtr")
]



In [17]:
# root directory to store output files inside
base_data_dir = Path("../data/unconstrained")

In [18]:
from diff_mfld.mfld import Mfld
import itertools

# run the optimization scheme

cfgs = itertools.product(
    metric_funcs, max_domain_scaling, optimizers, methods
)
cfg_pbar = tqdm.tqdm(cfgs, desc="Configurations")

for (
        (metric_fn, metric_fn_label),
        scaling,
        ((optimizer, optimizer_cfg), optimizer_label),
        ((exp_method, log_method), method_label)
) in cfg_pbar:

    # setup the manifold configuration
    metric = MetricField(lambda x1, x2, x3: metric_fn(x1, x2, x3, scaling=scaling))
    conn = metric.christoffels()  # Levi-Civita connection (metric-compatible and torsion-free)
    mfld = Mfld(metric, conn)

    compute_mfld = ComputeMfld(
        mfld=mfld,
        exp_method=exp_method,
        log_method=log_method,
        dist_method=log_method,  # which logarithm to use when computing distance
    )

    # scale the problem coordinates accordingly
    trials_p_starting = p_rand_starting * scaling
    trials_cost_center = cost_center * scaling

    # print(f"trials_cost_center: {trials_cost_center}")
    # assert False

    # create the problem
    cost, _ = create_problem(torch.tensor(trials_cost_center))

    trials_pbar = tqdm.tqdm(range(trials_p_starting.shape[0]), desc="Trials")
    for start_idx in trials_pbar:
        p0 = torch.tensor(trials_p_starting[start_idx])
        result = optimizer(cost, p0, compute_mfld, optimizer_cfg)
        print(result)

        # save the resulting dataclass
        filename = f"{optimizer_label}/metric_{metric_fn_label}__scaling_{scaling}__trial_{start_idx}__geod_method_{method_label}.pt"
        file_path = base_data_dir / filename
        print(f"\tSaving to {file_path}")

        torch.save(result, file_path)

Configurations: 0it [00:00, ?it/s]
Trials:   5%|▌         | 1/20 [00:01<00:35,  1.88s/it]

RiemGradDescentResult(success=True, p=tensor([-0.3172, -0.0939,  0.1706]), iters=8, history=RiemGradDescentHistory(p_hist=tensor([[ 0.0227, -0.3655, -0.2538],
        [-0.0884, -0.2767, -0.1151],
        [-0.1662, -0.2146, -0.0180],
        [-0.2207, -0.1711,  0.0500],
        [-0.2588, -0.1406,  0.0976],
        [-0.2855, -0.1193,  0.1309],
        [-0.3041, -0.1044,  0.1542],
        [-0.3172, -0.0939,  0.1706]]), f_hist=tensor([0.2193, 0.1075, 0.0527, 0.0258, 0.0126, 0.0062, 0.0030, 0.0015])))
	Saving to ../data/unconstrained/rgd/metric_euclid__scaling_1.0__trial_0__geod_method_ivpbvp.pt



Trials:  10%|█         | 2/20 [00:03<00:33,  1.86s/it]

RiemGradDescentResult(success=True, p=tensor([-0.3121, -0.1012,  0.1541]), iters=8, history=RiemGradDescentHistory(p_hist=tensor([[ 0.0846, -0.4542, -0.4536],
        [-0.0451, -0.3388, -0.2550],
        [-0.1359, -0.2580, -0.1159],
        [-0.1994, -0.2015, -0.0185],
        [-0.2439, -0.1619,  0.0496],
        [-0.2751, -0.1342,  0.0973],
        [-0.2969, -0.1148,  0.1307],
        [-0.3121, -0.1012,  0.1541]]), f_hist=tensor([0.3868, 0.1895, 0.0929, 0.0455, 0.0223, 0.0109, 0.0054, 0.0026])))
	Saving to ../data/unconstrained/rgd/metric_euclid__scaling_1.0__trial_1__geod_method_ivpbvp.pt



Trials:  15%|█▌        | 3/20 [00:05<00:31,  1.88s/it]

...

RiemTrustRegionResult(success=True, p=tensor([ 0.6807, -1.6512, -1.8025]), iters=2, history=RiemTrustRegionHistory(p_hist=tensor([[ 0.6807, -1.6512, -1.8025],
        [ 0.6807, -1.6512, -1.8025]]), f_hist=tensor([21.3015, 21.3015]), quality_hist=tensor([-1.3140, -1.2800]), radius_hist=tensor([0.0250, 0.0063])))
	Saving to ../data/unconstrained/rtr/metric_asymmetric__scaling_3.0__trial_17__geod_method_ivpbvp.pt



Trials:  95%|█████████▌| 19/20 [15:13<00:15, 15.64s/it]

RiemTrustRegionResult(success=True, p=tensor([ 0.2003, -1.2930, -1.7502]), iters=2, history=RiemTrustRegionHistory(p_hist=tensor([[ 0.2044, -1.2964, -1.7569],
        [ 0.2003, -1.2930, -1.7502]]), f_hist=tensor([36.6404, 36.5891]), quality_hist=tensor([0.1752, 0.2402]), radius_hist=tensor([0.0250, 0.0063])))
	Saving to ../data/unconstrained/rtr/metric_asymmetric__scaling_3.0__trial_18__geod_method_ivpbvp.pt



Trials: 100%|██████████| 20/20 [15:15<00:00, 45.77s/it]
Configurations: 24it [2:13:23, 333.50s/it]

RiemTrustRegionResult(success=True, p=tensor([ 1.0411, -1.4045, -1.2650]), iters=2, history=RiemTrustRegionHistory(p_hist=tensor([[ 1.0411, -1.4045, -1.2650],
        [ 1.0411, -1.4045, -1.2650]]), f_hist=tensor([12.4388, 12.4388]), quality_hist=tensor([-1.3852, -1.3428]), radius_hist=tensor([0.0250, 0.0063])))
	Saving to ../data/unconstrained/rtr/metric_asymmetric__scaling_3.0__trial_19__geod_method_ivpbvp.pt
